# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import optuna

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator
sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
from split_tools import get_train_test

sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from modelling4_utils import (
    StatsModelsEstimator,
    BetaVAEncoder,
    MLPipeline,
    OptunaOptimizer,
    deep_update,
    deep_update_dim_transformer_params,
    RANDOM_STATE,
)

seed = RANDOM_STATE

from catboost import CatBoostClassifier
from statsmodels.api import Logit

from sklearn.decomposition import PCA

from sklearn.metrics import (
    make_scorer,
    f1_score,
    mean_squared_error
)

import optuna
from functools import partial

from copy import deepcopy

from tqdm import trange

## Settings

In [3]:
scoring_metrics={
    'mse': mean_squared_error,
}
step_scoring_average = "mean"
n_trials = 100 # was 600

features_to_drop=(
    'sign_sedimentation_Re',
    'sign_sedimentation_Stk',
    'sign_particle_droplet_diameter_ratio',
)

save_model_and_metrics = True
metrics_file = "metrics_modelling4_7-dim_reduction.xlsx"

## Optimization function

In [4]:
def optimize_vae_optuna(
    dim_transformer:Type[BaseEstimator]=BetaVAEncoder,
    dim_transformer_params:dict=None,
    target:str='splashing', # Need for SMOTE before dim reduction
    dummy_estimator:Type[BaseEstimator]=CatBoostClassifier,  # placeholder
    objective:callable=None,
    direction:str="minimize",
    n_trials:int=n_trials,
    features_to_drop:tuple=features_to_drop,
    step_scoring_average:str=step_scoring_average,
    scoring_metrics:dict=scoring_metrics,
    verbose:bool=True,
    opt_cv_folds:int=5,
    seed:int=seed,
):
    dim_transformer_params = dim_transformer_params or {}
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=dummy_estimator,
        dim_transformer=dim_transformer,
        dim_transformer_params=dim_transformer_params,
        features_to_drop=features_to_drop,
        cv_folds=opt_cv_folds,
        verbose=verbose,
        step_scoring_average=step_scoring_average,
        scoring_metrics=scoring_metrics,
    )
    
    opt = OptunaOptimizer(
        objective=partial(
            objective,
            ml_pipe=ml_pipe,
        ),
        study_name='VAE_study',
        direction=direction,
        seed=seed,
    )
    
    opt.optimize(
        n_trials=n_trials,
        catch=(ValueError,),
    )
    
    best_params = opt.study.best_params
    
    # print('raw best_params')
    # display(best_params)
    
    return best_params


def VAE_objective(
    trial:optuna.trial.Trial,
    ml_pipe:MLPipeline,
):
    # VAE params
    # hidden_dim = 2**trial.suggest_int('log2_hidden_dim', 2, 8)
    hidden_dim = 2**trial.suggest_int('log2_hidden_dim', 2, 6)
    # normalization = trial.suggest_categorical('normalization', ['batch', 'layer', False])
    normalization = trial.suggest_categorical('normalization', ['batch', 'layer'])
    # activation = trial.suggest_categorical('activation', ['ReLU', 'LeakyReLU', False])
    activation = trial.suggest_categorical('activation', ['ReLU', 'LeakyReLU'])
    
    if activation == 'LeakyReLU':
        # negative_slope = trial.suggest_float('negative_slope', 0.01, .3, log=True)
        negative_slope = trial.suggest_float('negative_slope', 0.01, .1)
    else:
        negative_slope = None
    
    # BetaVAEncoder params
    # learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    beta_warmup_steps = trial.suggest_int('beta_warmup_steps', 100, 2000)
    # beta_start = trial.suggest_float('beta_start', 0.0, 0.5)
    # beta_end = trial.suggest_float('beta_end', 1.0, 10.0)
    beta_start = trial.suggest_float('beta_start', 0.0, 0.05)
    beta_end = trial.suggest_float('beta_end', 0.05, 0.3)
    
    suggested_transformer_params = {
        'VAE_params': {
            'hidden_dim': hidden_dim,
            'normalization': normalization,
            'activation': activation,
            'negative_slope': negative_slope,
        },
        'learning_rate': learning_rate,
        'beta_warmup_steps': beta_warmup_steps,
        'beta_start': beta_start,
        'beta_end': beta_end,
    }
    
    dim_transformer_params = deep_update_dim_transformer_params(
        ml_pipe=ml_pipe,
        suggested_params=suggested_transformer_params,
    )
    
    try:
        score = ml_pipe.step_transformer(
            dim_transformer_params=dim_transformer_params,
        )
    except ValueError as e:
        trial.set_user_attr('fail_reason', str(e))
        raise
    
    score_val = score[0]
    # score_train = score[1]
    return score_val


def get_vae_params(
    raw_vae_params:dict,
):
    hidden_dim = 2**raw_vae_params['log2_hidden_dim']
    suggested_transformer_params = {
        'VAE_params': {
            'hidden_dim': hidden_dim,
            'normalization': raw_vae_params['normalization'],
            'activation': raw_vae_params['activation'],
        },
        'learning_rate':    raw_vae_params['learning_rate'],
        'beta_warmup_steps': raw_vae_params['beta_warmup_steps'],
        'beta_start':       raw_vae_params['beta_start'],
        'beta_end':         raw_vae_params['beta_end'],
    }
    
    if 'negative_slope' in raw_vae_params:
        suggested_transformer_params['VAE_params']['negative_slope'] = (
            raw_vae_params['negative_slope']
        )
    
    return suggested_transformer_params

In [5]:
estimator_params_dict = {
    StatsModelsEstimator: {'model_class': Logit, 'verbose': False,},
    CatBoostClassifier: {'verbose': False,},
}

for latent_dim in trange(1, 8):
    print()
    print('-'*50)
    print(f'VAE with {latent_dim}-dimension latent space')
    print('-'*50)
    model_postfix = f'{latent_dim}-dim_vae'

    dim_transformer_params = {
        'VAE_params': {
                'latent_dim': latent_dim,
            },
        'verbose': False,
        'early_stopping': True,
        'early_stopping_patience': 10,
        'max_epochs': 500,
    }

    raw_vae_params = optimize_vae_optuna(
        dim_transformer_params=dim_transformer_params,
        objective=VAE_objective,
        verbose=False,
    )
    
    suggested_transformer_params = get_vae_params(
        raw_vae_params=raw_vae_params
    )

    dim_transformer_params = deepcopy(dim_transformer_params)

    deep_update(
        original=dim_transformer_params,
        updates=suggested_transformer_params,
    )
    dim_transformer_params['verbose'] = False
    print()
    print(f'{latent_dim}-dim_vae params:')
    display(dim_transformer_params)
    
    for target in ('splashing', 'no_fragmentation'):
        for estimator, estimator_params in estimator_params_dict.items():
            ml_pipe = MLPipeline(
                target=target,
                estimator=estimator,
                estimator_params=estimator_params,
                # dim_transformer=PCA,
                # dim_transformer_params={'n_components': 6},
                dim_transformer=BetaVAEncoder,
                dim_transformer_params=dim_transformer_params,
                features_to_drop=features_to_drop,
                metrics_file=metrics_file,
                model_postfix=model_postfix,
                verbose=False,
            )
            
            ml_pipe.run(
                save_model_and_metrics=True,
                verbose=False,
                cv_verbose=False,
            )

  0%|          | 0/7 [00:00<?, ?it/s]


--------------------------------------------------
VAE with 1-dimension latent space
--------------------------------------------------


[I 2025-05-07 22:21:32,865] A new study created in memory with name: VAE_study
[I 2025-05-07 22:21:38,472] Trial 0 finished with value: 0.8100192015946828 and parameters: {'log2_hidden_dim': 3, 'normalization': 'batch', 'activation': 'ReLU', 'learning_rate': 0.00020511104188433984, 'beta_warmup_steps': 210, 'beta_start': 0.04330880728874676, 'beta_end': 0.20027875293580222}. Best is trial 0 with value: 0.8100192015946828.
[I 2025-05-07 22:21:51,627] Trial 1 finished with value: 0.7155476615699805 and parameters: {'log2_hidden_dim': 5, 'normalization': 'layer', 'activation': 'ReLU', 'learning_rate': 0.0002310201887845295, 'beta_warmup_steps': 448, 'beta_start': 0.015212112147976888, 'beta_end': 0.18118910790805948}. Best is trial 1 with value: 0.7155476615699805.
[I 2025-05-07 22:22:07,071] Trial 2 finished with value: 0.5637039654487207 and parameters: {'log2_hidden_dim': 4, 'normalization': 'layer', 'activation': 'LeakyReLU', 'negative_slope': 0.04297256589643226, 'learning_rate': 0.0


1-dim_vae params:


{'VAE_params': {'latent_dim': 1,
  'hidden_dim': 64,
  'normalization': 'layer',
  'activation': 'ReLU'},
 'verbose': False,
 'early_stopping': True,
 'early_stopping_patience': 10,
 'max_epochs': 500,
 'learning_rate': 0.009123725376946416,
 'beta_warmup_steps': 1515,
 'beta_start': 0.02877368219371967,
 'beta_end': 0.14426114608260293}

...


7-dim_vae params:


{'VAE_params': {'latent_dim': 7,
  'hidden_dim': 64,
  'normalization': 'batch',
  'activation': 'ReLU'},
 'verbose': False,
 'early_stopping': True,
 'early_stopping_patience': 10,
 'max_epochs': 500,
 'learning_rate': 0.006912175658094458,
 'beta_warmup_steps': 1441,
 'beta_start': 0.0010174512422593356,
 'beta_end': 0.07352336628703057}

Model saved in ..\results\models_modelling4\Logit_splashing_smote_7-dim_vae
Model saved in ..\results\models_modelling4\CatBoostClassifier_splashing_smote_7-dim_vae
Model saved in ..\results\models_modelling4\Logit_no_fragmentation_smote_7-dim_vae


100%|██████████| 7/7 [2:55:43<00:00, 1506.15s/it]

Model saved in ..\results\models_modelling4\CatBoostClassifier_no_fragmentation_smote_7-dim_vae
